In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("clean_transactions.csv", dtype={"Invoice": str, "StockCode": str})
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Customer ID"] = df["Customer ID"].astype("Int64")

print(f"Jumlah baris: {len(df):,}")
print(f"Customer ID kosong: {df['Customer ID'].isna().sum():,}")

Jumlah baris: 1,027,069
Customer ID kosong: 229,200


In [2]:
# Kalau dalam 1 invoice ada baris yang punya Customer ID, pakai ID itu
sebelum = df["Customer ID"].isna().sum()
df["Customer ID"] = df["Customer ID"].fillna(df.groupby("Invoice")["Customer ID"].transform("max"))
print(f"Diisi dari invoice yang sama: {sebelum - df['Customer ID'].isna().sum():,}")

# Sisanya guest: 1 invoice = 1 guest, ID mulai 100001 urut tanggal
df["is_guest"] = df["Customer ID"].isna()
guest = (df[df["is_guest"]]
         .groupby("Invoice")["InvoiceDate"].min()
         .reset_index()
         .sort_values(["InvoiceDate", "Invoice"]))
guest["guest_id"] = range(100001, 100001 + len(guest))

df = df.merge(guest[["Invoice", "guest_id"]], on="Invoice", how="left")
df["Customer ID"] = df["Customer ID"].fillna(df["guest_id"].astype("Int64")).astype(int)
df = df.drop(columns="guest_id")

print(f"Invoice guest: {len(guest):,}")
print(f"Customer ID kosong sekarang: {df['Customer ID'].isna().sum()}")

Diisi dari invoice yang sama: 0
Invoice guest: 3,497
Customer ID kosong sekarang: 0


In [3]:
nama_depan = [
    "James","Olivia","Liam","Emma","Noah","Sophia","Lucas","Mia","Ethan","Amelia",
    "Daniel","Chloe","Henry","Isabella","Leo","Grace","Oscar","Hannah","Felix","Clara",
    "Thomas","Emily","William","Charlotte","Jack","Lily","Samuel","Ella","Benjamin","Ava",
    "Max","Lena","Jonas","Anna","Paul","Marie","Hugo","Camille","Louis","Julia",
    "Marco","Giulia","Luca","Martina","Mateo","Lucia","Diego","Elena","Erik","Freya",
    "Lars","Ingrid","Jan","Eva","Pedro","Ines","Adam","Nora","Victor","Alice"]

nama_belakang = [
    "Smith","Johnson","Brown","Taylor","Wilson","Martin","Muller","Dubois","Rossi","Garcia",
    "Andersen","Novak","Silva","Jensen","Schmidt","Moreau","Ricci","Lopez","Nielsen","Walker",
    "Davies","Evans","Thomas","Roberts","Wright","Green","Hall","Wood","Clarke","Hughes",
    "Edwards","Turner","Hill","Cooper","Ward","Morris","King","Baker","Harris","Lewis",
    "Schneider","Fischer","Weber","Meyer","Wagner","Becker","Hoffmann","Koch","Richter","Klein",
    "Bernard","Petit","Durand","Leroy","Lambert","Fontaine","Girard","Bonnet","Mercier","Laurent",
    "Bianchi","Romano","Colombo","Bruno","Ferrari","Esposito","Greco","Marino","Conti","Gallo",
    "Martinez","Sanchez","Perez","Gomez","Fernandez","Ruiz","Diaz","Moreno","Alvarez","Romero",
    "Hansen","Larsen","Olsen","Pedersen","Johansson","Lindberg","Berg","Virtanen","Korhonen","Nowak",
    "Kowalski","Horvat","Costa","Santos","Ferreira","Pereira","Murphy","Kelly","Byrne","Walsh"]

cust = df.groupby("Customer ID", as_index=False)["is_guest"].first()
ids  = cust["Customer ID"]
asli = ~cust["is_guest"]

cust["customer_name"] = np.where(
    asli,
    [f"{nama_depan[i % 60]} {nama_belakang[(i // 60) % 100]}" for i in ids],
    "John Doe")

cust["customer_email"] = np.where(
    asli,
    cust["customer_name"].str.lower().str.replace(" ", ".") + "." + ids.astype(str) + "@example.com",
    "john.doe@gmail.com")

cust["customer_phone"] = np.where(
    asli,
    "+447" + ids.astype(str) + (ids * 37 % 1000).astype(str).str.zfill(3),
    None)

df = df.merge(cust.drop(columns="is_guest"), on="Customer ID", how="left")
print(cust.sample(8, random_state=1).to_string(index=False))

 Customer ID  is_guest customer_name                  customer_email customer_phone
      101441      True      John Doe              john.doe@gmail.com            NaN
       13653     False     Anna Wood     anna.wood.13653@example.com   +44713653161
       14934     False Pedro Richter pedro.richter.14934@example.com   +44714934558
      101126      True      John Doe              john.doe@gmail.com            NaN
      103392      True      John Doe              john.doe@gmail.com            NaN
      100891      True      John Doe              john.doe@gmail.com            NaN
       18027     False    Ella Smith    ella.smith.18027@example.com   +44718027999
       13008     False    Erik Ricci    erik.ricci.13008@example.com   +44713008296


In [4]:
c_asli  = cust[~cust["is_guest"]]
c_guest = cust[cust["is_guest"]]

print(f"Baris akhir: {len(df):,}")
print(f"Customer asli: {len(c_asli):,} | Guest: {len(c_guest):,}")
print("Customer ID kosong:", df["Customer ID"].isna().sum())
print("Nama bukan 2 kata:", (cust["customer_name"].str.split().str.len() != 2).sum())
print("Nama asli kembar:", c_asli["customer_name"].duplicated().sum())
print("Email asli kembar:", c_asli["customer_email"].duplicated().sum())
print("Phone asli kembar:", c_asli["customer_phone"].duplicated().sum())
print("Phone terpanjang:", c_asli["customer_phone"].str.len().max(), "karakter")

Baris akhir: 1,027,069
Customer asli: 5,940 | Guest: 3,497
Customer ID kosong: 0
Nama bukan 2 kata: 0
Nama asli kembar: 0
Email asli kembar: 0
Phone asli kembar: 0
Phone terpanjang: 12 karakter


In [ ]:
df["is_guest"] = df["is_guest"].astype(int)
df.to_csv("formatted_transactions.csv", index=False, encoding="utf-8", lineterminator="\n")
print("Tersimpan: formatted_transactions.csv")
print(df.columns.tolist())

Tersimpan: enriched_transactions.csv
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'invoice_type', 'product_type', 'is_guest', 'customer_name', 'customer_email', 'customer_phone']
